<a href="https://colab.research.google.com/github/abubakirovx/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
print("Menu Detector")

Menu Detector


In [17]:
# IMPORT LIBRARIES
from google.colab import drive

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import mobilenet_v2
import torchvision.transforms as transforms
from torch.utils.data import Dataset,DataLoader

from PIL import Image, UnidentifiedImageError
import os
import numpy as np



In [18]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
#Define Dataset Path
DATASET_PATH = "/content/drive/MyDrive/food101"
print("Dataset Path",DATASET_PATH)

Dataset Path /content/drive/MyDrive/food101


In [20]:
CUSTOM_CLASS_MAPPING={
    "pizza":"pizza",
    "hot_dog":"hot_dog",
    "osh":"pilaf",
    "kabob":"kebab",
    "manti":"manti",
    "chocolate_cake":"desert", # label grouping |class consolidation
    "cheesecake":"desert"      # label grouping |class consolidation
}

CLASSES =['pizza','hot_dog','pilaf','kebab','desert','manti']
CLASS_TO_IDX={cls:i for i,cls in enumerate(CLASSES)}
NUM_CLASSES=len(CLASSES)

print(CLASS_TO_IDX)
print(NUM_CLASSES)

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

# 0 ~ 255 ToTensor => 0.0 ~ 0.1 | 0.234 | H, W, C => C, H, W
# Normalize => pixel = (pixel-mean) / std


{'pizza': 0, 'hot_dog': 1, 'pilaf': 2, 'kebab': 3, 'desert': 4, 'manti': 5}
6


In [21]:
# Custom Dataset Class

class FoodDataset(Dataset):
  def __init__(self,images,labels,transform=None):
    self.images=images
    self.labels=labels
    self.transform=transform

  def __len__ (self):
    print("images length",len(self.images))
    return len(self.images)

  def __getitem__(self,idx):
    image_path = self.images[idx]
    print("images image_path",image_path)
    label=self.labels[idx]
    print("images label",label)
    try:
      image=Image.open(image_path).convert('RGB')

    except(UnidentifiedImageError,OSError):
        print(f"Skipping broken image: {image_path}")
        return self.__getitem__((idx+1)%len(self.images))
    if self.transform:
         image = self.transform(image)
    return image ,label

In [22]:
# Gather and Split Data

all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
  class_path=os.path.join(DATASET_PATH,original_class) #/content/drive/MyDrive/food101/pizza
  print("class_path:",class_path)
  if not os.path.exists(class_path):
      print(f"Warning: {class_path} is not found!")
      continue
  for img in os.listdir(class_path):
    if img.endswith(('.jpg','.jpeg','.png')):
      full_path=os.path.join(class_path,img) # /content/drive/MyDrive/food101/pizza/3102271.jpg
      all_images.append((full_path,CLASS_TO_IDX[mapped_class])) # ('/content/drive/MyDrive/food101/pizza/3102271.jpg', 0)
np.random.shuffle(all_images)
split=int(0.8*len(all_images))
train_data=all_images[:split]
val_data=all_images[split:]



tran_images,train_labels = zip(*train_data)
val_images,val_labels=zip(*val_data)

# print("all_images:",all_images)

dataset = FoodDataset(tran_images,train_labels)
print(len(dataset))
img,lbl=dataset[0]

class_path: /content/drive/MyDrive/food101/pizza
class_path: /content/drive/MyDrive/food101/hot_dog
class_path: /content/drive/MyDrive/food101/osh
class_path: /content/drive/MyDrive/food101/kabob
class_path: /content/drive/MyDrive/food101/manti
class_path: /content/drive/MyDrive/food101/chocolate_cake
class_path: /content/drive/MyDrive/food101/cheesecake
images length 3440
3440
images image_path /content/drive/MyDrive/food101/pizza/928670.jpg
images label 0


In [23]:
train_dataset=FoodDataset(train_data,train_labels,transform=transform)
val_dataset=FoodDataset(val_data,val_labels,transform=transform)


In [24]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,num_workers=2) #num_workers is like thread |parallel loading for speed
val_dataset = DataLoader(val_dataset,batch_size=32,shuffle=False,num_workers=2)

images length 3440
images length 3440


In [25]:
# pretrained model
model= mobilenet_v2(weights="IMAGENET1K_V1") #pretained_model | lightweight | CNN | 1000 class | million
model.classifier[1]=nn.Linear(model.classifier[1].in_features,NUM_CLASSES) # fine_tuning | backbone | model layer freeze


In [26]:
device = torch.device('cuda' if torch.cuda.is_available()else "cpu")
print("device:",device)
model=model.to(device)

device: cuda


In [29]:
criterion = nn.CrossEntropyLoss() # Loss Function | 70% burger | 30% pilaf
optimizer = optim.Adam(model.parameters(),lr=0.001) # weight
torch.backends.cudnn.benchmark = True # Benchmark Setting | Trick | 10-20% more speed
